# Accelerated "deconvolution" of projected images using CARE

This notebook demonstrates applying a CARE model for a 2D denoising task, assuming that training was already completed via [2_training.ipynb](2_training.ipynb), or that a pre-trained model is available.
 
**We provide a pre-trained model**, which has been trained of an heterogenous set of ISS data coming from 2 different microscopes (Leica and Zeiss in our lab), magnifications (20x and 40x) and tissues (fly embryos and ovaries, developing human spinal cord and mouse brains). 

The model is available at `https://github.com/Moldia/ISS2025/tree/main/ISS_CARE/models/spots`.

Our models have been benchmarked using chicken brain data (never seen in the training process), and show excellent performance and a great speed tradeoff Vs canonical deconvolution. 

**However, we strongly advise training your own model, especially if you don't belong to Mats Nilsson's lab. Your microscope and data might be significantly different from ours, and our models could potentially produce artifacts in your data.** 

More documentation about how CARE works, and about how to train your own models, is available at http://csbdeep.bioimagecomputing.com/doc/.

In our lab's workflow, CARE-mediate deconvolution should be applied just after the preprocessing step (without deconvolution; `deconvolution_method = None`). 

For this notebook to work correctly, you should have a specific folder for each one of the regions you want to process, `/R1/`, `/R2/`, etc. Each region folder should contain the following subfolders tree: `/preprocessing/CycleX/4_retiled/`. Under `/preprocessing/` you might still have other subfolders such as `/1_mipped/`, `/2_ome_tiffs/` and `/3_stitched/`, but these are irrelevant at this stage.

The important thing is that you have a `/preprocessing/CycleX/4_retiled/`, because the retiled images are the starting point of the CARE-mediate deconvolution process.




## We start by importing the necessary modules

In [ ]:
from ISS_CARE.ISS_CARE_prediction import ISS_CARE_predict, visualize_random_care_predictions

## Denoising prediction with CARE

### Core parameters
`input_dir` (str): Path to parent directory containing region folders (`R1/`, `R2/`, …).  
`regions_to_process` (list[int] | None): 1-based region indices; `None` means all detected regions are processed.  
`output_dir_prefix` (str | None): Optional output root. If `None`, outputs are saved inside the input structure at `input_dir/R#/preprocessing/CycleX/4_retiled/CARE/`. If set, outputs are saved to `output_dir_prefix/R#/preprocessing/CycleX/4_retiled/CARE/`.  
`model_dir` (str | Path): Directory containing trained CARE models.  
`model_name` (str): Model subfolder name.  
`dapi_ch` (int, zero-based): DAPI channel index. Files matching `*_ch{dapi_ch}.tif` are copied unchanged; all other channels are denoised with CARE.

### Normalization behavior
`normalize_input` (bool | None, default=`None`): Controls whether percentile normalization is applied before prediction.  
- `None` → automatically check `training_metadata.json` and follow training behavior (recommended)  
- `False` → use raw intensities (raw-intensity behavior)  
- `True` → apply percentile normalization before prediction  

`normalization_pmin`, `normalization_pmax`, `normalization_eps`: Optional normalization parameters. If `None`, values are taken from training metadata if available, otherwise defaults (`1.0`, `99.8`, `1e-8`) are used.

### Output scaling
`output_rescale_mode` (str, default=`"none"`):  
- `"none"` → **recommended default** (keeps prediction in model output scale)  
- `"percentile"` → optional fallback if output looks too dim   

### Training metadata auto-detection
The function looks for:  
`model_dir/model_name/training_metadata.json`

If found, it can automatically determine:
- whether normalization was used during training  
- normalization parameters (`PMIN`, `PMAX`, `EPS`)  

If missing, it safely falls back to raw inference unless overridden.

### Debugging
`debug_prints` (bool, default=`True`): Print intensity statistics.  
`debug_print_limit` (int, default=`3`): Number of images per cycle to print debug stats for.  

Debug output helps detect:
- wrong normalization  
- saturated outputs  
- extremely dim predictions  

### Output behavior
- Outputs are saved as `uint16` TIFFs  
- DAPI channels are copied unchanged  
- Existing valid outputs are skipped unless `overwrite=True`  
- CSV metadata is preserved  
- XML provenance is written only if new outputs are generated  

### Visualization (after prediction)
Use:
`visualize_random_care_predictions(...)`

This randomly samples raw/prediction pairs and displays:
- raw image  
- CARE output  
- intensity histogram  



In [ ]:
input_dir = '/path/to/regions/'
model_dir = '/path/to/model/' # /home/<user>/ISS2025/ISS_CARE/models/
model_name = 'spots'

In [ ]:
ISS_CARE_predict(
    input_dir=input_dir,
    model_dir=model_dir,
    model_name=model_name,
    dapi_ch=4,
    output_dir_prefix=None,
    normalize_input=False,
    output_rescale_mode="none",
    overwrite=False,
)

In [ ]:
_ = visualize_random_care_predictions(
    input_dir=input_dir,
    output_dir_prefix=output_dir_prefix,
    n_pairs=5,          
    dapi_ch=4,          
)